In [ ]:
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import re
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv("C:/Users/leila/Dropbox/MayoWetlands/wetland_alldata_2025_only_uscore_averaged.csv")

# Strip whitespace from column names
df.columns = df.columns.str.strip()

print(f"Loaded dataset: {df.shape}")
print(f"Classes: {df['Class'].value_counts()}")

include_upland = False

# ===========================================================
# 3) Filter data based on mode
# ===========================================================

if include_upland:
    df_analysis = df.copy()
    print("\nMode: INCLUDING UPLAND (no chem, no field veg, no hydrology)")
else:
    df_analysis = df[df['Class'] != 'UL'].copy()
    print("\nMode: EXCLUDING UPLAND (includes chem + field veg)")

print(f"Analysis dataset: {df_analysis.shape}")

# Satellite remote sensing
sat_start = df_analysis.columns.get_loc("NDVI_amp_harmonic")
sat_end   = df_analysis.columns.get_loc("TCW_p90") + 1
sat_cols  = df_analysis.columns[sat_start:sat_end].tolist()

# Topography
topo_start = df_analysis.columns.get_loc("SWI_median")
topo_end   = df_analysis.columns.get_loc("Geom_mode_10") + 1
topo_cols  = df_analysis.columns[topo_start:topo_end].tolist()

# UAV multispectral indices
uav_ms_start = df_analysis.columns.get_loc("SR_MEAN")
uav_ms_end   = df_analysis.columns.get_loc("NIR_stdev") + 1
uav_ms_cols  = df_analysis.columns[uav_ms_start:uav_ms_end].tolist()


# UAV structure
uav_struct_start = df_analysis.columns.get_loc("average_veg_height")
uav_struct_end   = df_analysis.columns.get_loc("stem_density") + 1
uav_struct_cols  = df_analysis.columns[uav_struct_start:uav_struct_end].tolist()

# Field veg (conditionally included)
if not include_upland:
    field_start = df_analysis.columns.get_loc("tall_tree")
    field_end   = df_analysis.columns.get_loc("equisetum_dom") + 1
    field_cols  = df_analysis.columns[field_start:field_end].tolist()
else:
    field_cols = []

# Chemistry (conditionally included)
if not include_upland:
    chem_start = df_analysis.columns.get_loc("Al")
    chem_end   = df_analysis.columns.get_loc("ORPmV") + 1
    chem_cols  = df_analysis.columns[chem_start:chem_end].tolist()
else:
    chem_cols = []

# ===========================================================
# 6) Combine all predictor columns
# ===========================================================

all_predictor_cols = sat_cols + topo_cols + uav_ms_cols + uav_struct_cols + field_cols + chem_cols

# Create group mapping
group_map = {}
group_map.update({col: 'Satellite' for col in sat_cols})
group_map.update({col: 'Topography' for col in topo_cols})
group_map.update({col: 'UAV_MS' for col in uav_ms_cols})
group_map.update({col: 'UAV_Structure' for col in uav_struct_cols})
if not include_upland:
    group_map.update({col: 'Field_Veg' for col in field_cols})
    group_map.update({col: 'Chemistry' for col in chem_cols})

# Define group names and order
if include_upland:
    group_names = ['Satellite', 'Topography', 'UAV_MS', 'UAV_Structure']
else:
    group_names = ['Satellite', 'Topography', 'UAV_MS', 'UAV_Structure', 'Field_Veg', 'Chemistry']

print(f"\n" + "="*60)
print("INITIAL VARIABLE SET")
print("="*60)
print(f"Total predictor variables: {len(all_predictor_cols)}")
print(f"Groups: {group_names}")

In [ ]:
# ===========================================================
# 7) Prepare data for modeling
# ===========================================================

print("\n" + "="*60)
print("PREPARING DATA FOR MODELING")
print("="*60)

X = df_analysis[all_predictor_cols].copy()
y = df_analysis['Class'].copy()

# Handle NAs (drop rows with any NA in predictors)
mask = ~X.isna().any(axis=1)
X = X[mask]
y = y[mask]

print(f"\nFinal dataset:")
print(f"  Samples: {len(X)}")
print(f"  Variables: {len(X.columns)}")
print(f"  Class distribution:\n{y.value_counts()}")

# ===========================================================
# 8) Train Random Forest
# ===========================================================

print("\n" + "="*60)
print("TRAINING RANDOM FOREST CLASSIFIER")
print("="*60)

RF = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
RF.fit(X, y)

print(f"\nRandom Forest accuracy: {RF.score(X, y):.3f}")
print(f"Classes: {RF.classes_}")

# ===========================================================
# 9) Compute SHAP values
# ===========================================================

print("\n" + "="*60)
print("COMPUTING SHAP VALUES")
print("="*60)

explainer = shap.TreeExplainer(RF)
shap_values_raw = explainer.shap_values(X)

# Handle both list and 3D array formats
if isinstance(shap_values_raw, list):
    shap_values = shap_values_raw
else:
    # Convert 3D array to list of 2D arrays
    shap_values = [shap_values_raw[:, :, i] for i in range(shap_values_raw.shape[2])]

print(f"\nSHAP values computed:")
print(f"  Number of classes: {len(shap_values)}")
print(f"  SHAP shape per class: {shap_values[0].shape}")


In [ ]:
# ===========================================================
# 10) Group SHAP values by variable groups
# ===========================================================

print("\n" + "="*60)
print("GROUPING SHAP VALUES")
print("="*60)

n_classes = len(shap_values)

grouped_class_mats = []
for c in range(n_classes):
    class_shap = shap_values[c]  # shape: (n_samples, n_features)
    group_sums = []
    
    for gname in group_names:
        # Get indices for this group
        group_indices = [i for i, col in enumerate(X.columns) if group_map[col] == gname]
        # Sum absolute SHAP values across features in this group
        group_sum = np.abs(class_shap[:, group_indices]).sum(axis=1)
        group_sums.append(group_sum)
    
    # Stack into matrix: (n_samples, n_groups)
    grouped_class_mats.append(np.column_stack(group_sums))

print("Group-level SHAP matrices created")

# ===========================================================
# 11) Setup colors for classes
# ===========================================================

if include_upland:
    classlist = ['Bog', 'Fen', 'Marsh', 'Swamp', 'UL']
    colors = ['#4C0073', '#FFFF00', '#E64C00', '#5E4037', '#8c564b']
else:
    classlist = ['Bog', 'Fen', 'Marsh', 'Swamp']
    colors = ['#4C0073', '#FFFF00', '#E64C00', '#5E4037']

cmap = matplotlib.colors.ListedColormap(colors)

# ===========================================================
# 12) Plot grouped SHAP summary (stacked by class)
# ===========================================================

print("\n" + "="*60)
print("PLOTTING GROUP-LEVEL SHAP IMPORTANCE")
print("="*60)

model_classes = list(RF.classes_)
name_to_idx   = {n: i for i, n in enumerate(classlist)}
legend_names  = [mc if mc in name_to_idx else str(mc) for mc in model_classes]
legend_colors = [colors[name_to_idx[mc]] if mc in name_to_idx else '#999999'
                 for mc in model_classes]
cmap_ordered  = matplotlib.colors.ListedColormap(legend_colors)

plt.figure(figsize=(9, 3.6))
shap.summary_plot(
    grouped_class_mats,
    features=group_names,
    class_names=legend_names,
    color=cmap_ordered,
    class_inds='original',
    plot_type='bar',
    show=False
)
plt.xlabel('mean(|SHAP value|) (average impact on model output magnitude)')
title_suffix = "(with Upland)" if include_upland else "(wetlands only)"
plt.title(f'Group-level SHAP importance {title_suffix}')
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================
# 13) Individual group SHAP analysis
# ===========================================================

print("\n" + "="*60)
print("INDIVIDUAL VARIABLE SHAP ANALYSIS BY GROUP")
print("="*60)

# Create a dictionary mapping group names to their column lists
group_cols_dict = {
    'Satellite': sat_cols,
    'Topography': topo_cols,
    'UAV_MS': uav_ms_cols,
    'UAV_Structure': uav_struct_cols
}

if not include_upland:
    group_cols_dict['Field_Veg'] = field_cols
    group_cols_dict['Chemistry'] = chem_cols

# Loop through each group
for group_name in group_names:
    print(f"\n{'='*60}")
    print(f"Processing group: {group_name}")
    print(f"{'='*60}")
    
    # Get columns for this group
    group_cols = group_cols_dict[group_name]
    
    if len(group_cols) == 0:
        print(f"No variables in {group_name} - skipping")
        continue
    
    # Prepare data for this group only
    X_group = df_analysis[group_cols].copy()
    y_group = df_analysis['Class'].copy()
    
    # Handle NAs
    mask = ~X_group.isna().any(axis=1)
    X_group = X_group[mask]
    y_group = y_group[mask]
    
    print(f"Variables in group: {len(group_cols)}")
    print(f"Samples after NA removal: {len(X_group)}")
    
    # Train Random Forest for this group
    RF_group = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )
    RF_group.fit(X_group, y_group)
    
    print(f"Random Forest accuracy: {RF_group.score(X_group, y_group):.3f}")
    
    # Compute SHAP values
    explainer_group = shap.TreeExplainer(RF_group)
    shap_values_raw_group = explainer_group.shap_values(X_group)
    
    # Handle both list and 3D array formats
    if isinstance(shap_values_raw_group, list):
        shap_values_group = shap_values_raw_group
    else:
        shap_values_group = [shap_values_raw_group[:, :, i] for i in range(shap_values_raw_group.shape[2])]
    
    # Create color map for this model's class order
    model_classes = list(RF_group.classes_)
    name_to_idx = {n: i for i, n in enumerate(classlist)}
    legend_names = [mc if mc in name_to_idx else str(mc) for mc in model_classes]
    legend_colors = [colors[name_to_idx[mc]] if mc in name_to_idx else '#999999'
                     for mc in model_classes]
    cmap_group = matplotlib.colors.ListedColormap(legend_colors)
    
    # Plot SHAP summary for individual variables
    plt.figure(figsize=(9, 6))
    shap.summary_plot(
        shap_values_group,
        features=X_group,
        feature_names=X_group.columns.tolist(),
        class_names=legend_names,
        color=cmap_group,
        class_inds='original',
        plot_type='bar',
        show=False,
        max_display=min(20, len(group_cols))
    )
    plt.xlabel('mean(|SHAP value|) (average impact on model output magnitude)')
    title_suffix = "(with Upland)" if include_upland else "(wetlands only)"
    plt.title(f'{group_name} - Variable-level SHAP importance {title_suffix}')
    plt.tight_layout()
    plt.show()
    
    print(f"Completed {group_name}")


In [ ]:
# ===========================================================
# 14) Variable reduction within each group using SHAP
# ===========================================================

print("\n" + "="*60)
print("VARIABLE REDUCTION BASED ON SHAP VALUES")
print("="*60)

# Store results for each group
reduction_results = []
all_keep_features = []
all_drop_features = []

# Loop through each group to reduce variables
for group_name in group_names:
    print(f"\n{'='*60}")
    print(f"Reducing variables in: {group_name}")
    print(f"{'='*60}")
    
    # Get columns for this group
    group_cols = group_cols_dict[group_name]
    
    if len(group_cols) == 0:
        print(f"No variables in {group_name} - skipping")
        continue
    
    # *** SKIP REDUCTION FOR UAV_STRUCTURE - KEEP ALL ***
    if group_name == 'UAV_Structure':
        print(f"KEEPING ALL UAV_Structure variables (no reduction)")
        print(f"Variables kept: {len(group_cols)}")
        
        all_keep_features.extend(group_cols)
        
        reduction_results.append({
            'Group': group_name,
            'Original_vars': len(group_cols),
            'Kept_vars': len(group_cols),
            'Dropped_vars': 0,
            'Full_OA': 1.0,
            'Reduced_OA': 1.0,
            'OA_change': 0.0
        })
        
        print(f"\nCompleted {group_name} (all kept)")
        continue
    
    # *** NORMAL REDUCTION FOR OTHER GROUPS ***
    
    # Prepare data for this group only
    X_group = df_analysis[group_cols].copy()
    y_group = df_analysis['Class'].copy()
    
    # Handle NAs
    mask = ~X_group.isna().any(axis=1)
    X_group = X_group[mask]
    y_group = y_group[mask]
    
    print(f"Original variables: {len(group_cols)}")
    print(f"Samples: {len(X_group)}")
    
    # Train Random Forest for this group
    RF_group = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )
    RF_group.fit(X_group, y_group)
    
    oa_full = RF_group.score(X_group, y_group)
    print(f"Full model OA: {oa_full:.3f}")
    
    # Compute SHAP values
    explainer_group = shap.TreeExplainer(RF_group)
    shap_values_raw_group = explainer_group.shap_values(X_group)
    
    # Handle both list and 3D array formats
    if isinstance(shap_values_raw_group, list):
        shap_values_group = shap_values_raw_group
    else:
        shap_values_group = [shap_values_raw_group[:, :, i] for i in range(shap_values_raw_group.shape[2])]
    
    # Calculate mean absolute SHAP value for each feature (averaged across all classes)
    mean_abs_shap = np.zeros(len(X_group.columns))
    for class_shap in shap_values_group:
        mean_abs_shap += np.abs(class_shap).mean(axis=0)
    mean_abs_shap /= len(shap_values_group)
    
    # Create importance dataframe
    shap_importance = pd.DataFrame({
        'Feature': X_group.columns,
        'MeanAbsSHAP': mean_abs_shap
    }).sort_values('MeanAbsSHAP', ascending=False).reset_index(drop=True)
    
    # Calculate relative importance and cumulative percentage
    shap_importance['RelToMax_pct'] = (shap_importance['MeanAbsSHAP'] / shap_importance['MeanAbsSHAP'].max()) * 100
    shap_importance['Cumulative_pct'] = (shap_importance['MeanAbsSHAP'].cumsum() / shap_importance['MeanAbsSHAP'].sum()) * 100
    
    print("\n=== SHAP Importance Ranking ===")
    print(shap_importance.to_string(index=False))
    
    # *** USE 90% CUMULATIVE THRESHOLD FOR ALL GROUPS (EXCEPT UAV_STRUCTURE) ***
    threshold = 90
    print(f"\nUsing {threshold}% cumulative threshold")
    keep_features = shap_importance.loc[
        shap_importance["Cumulative_pct"] <= threshold,
        "Feature"
    ].tolist()
    
    drop_features = [f for f in X_group.columns if f not in keep_features]
    
    print(f"\n=== Reduction Summary ===")
    print(f"Kept: {len(keep_features)} / {len(group_cols)} variables")
    print(f"Dropped: {len(drop_features)} variables")
    
    if len(drop_features) > 0:
        print(f"\nDropped variables:")
        for feat in drop_features:
            row = shap_importance[shap_importance['Feature'] == feat].iloc[0]
            print(f"  - {feat}: {row['MeanAbsSHAP']:.4f} (rank: {row.name + 1}, cumulative: {row['Cumulative_pct']:.1f}%)")
    
    # Test reduced model
    if len(keep_features) > 0 and len(keep_features) < len(group_cols):
        X_group_reduced = X_group[keep_features]
        RF_reduced = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
        RF_reduced.fit(X_group_reduced, y_group)
        oa_reduced = RF_reduced.score(X_group_reduced, y_group)
        
        print(f"\n=== Model Comparison ===")
        print(f"Full model OA:    {oa_full:.3f}")
        print(f"Reduced model OA: {oa_reduced:.3f}")
        print(f"Change in OA:     {oa_reduced - oa_full:+.3f}")
    else:
        oa_reduced = oa_full
        print("\nNo reduction performed (all variables kept)")
    
    # Store results
    reduction_results.append({
        'Group': group_name,
        'Original_vars': len(group_cols),
        'Kept_vars': len(keep_features),
        'Dropped_vars': len(drop_features),
        'Full_OA': oa_full,
        'Reduced_OA': oa_reduced,
        'OA_change': oa_reduced - oa_full
    })
    
    all_keep_features.extend(keep_features)
    all_drop_features.extend(drop_features)

# ===========================================================
# 15) Summary of all reductions
# ===========================================================

print("\n" + "="*60)
print("SUMMARY: Variable Reduction Across All Groups")
print("="*60)

results_df = pd.DataFrame(reduction_results)
print("\n" + results_df.to_string(index=False))

print(f"\n=== Overall Summary ===")
print(f"Total original variables: {len(all_predictor_cols)}")
print(f"Total kept variables: {len(all_keep_features)}")
print(f"Total dropped variables: {len(all_drop_features)}")

print(f"\n=== All Dropped Variables (n={len(all_drop_features)}) ===")
for feat in sorted(all_drop_features):
    print(f"  - {feat}")

In [ ]:
# ===========================================================
# 16) Apply variable reductions and save reduced dataset
# ===========================================================

print("\n" + "="*60)
print("APPLYING VARIABLE REDUCTIONS AND SAVING")
print("="*60)

# Create reduced dataframe by dropping all identified low-importance variables
df_reduced = df_analysis.drop(columns=all_drop_features)

print(f"\nDataframe shapes:")
print(f"  Original: {df.shape}")
print(f"  After u-score: {df_analysis.shape}")
print(f"  After reduction: {df_reduced.shape}")

# Show what was dropped per group
print("\n=== Variables Dropped Per Group ===")
for group_name in group_names:
    group_cols = group_cols_dict.get(group_name, [])
    dropped_in_group = [f for f in all_drop_features if f in group_cols]
    if len(dropped_in_group) > 0:
        print(f"\n{group_name} (dropped {len(dropped_in_group)}):")
        for feat in dropped_in_group:
            print(f"  - {feat}")
    else:
        print(f"\n{group_name}: No variables dropped (all kept)")

# Test overall model performance with reduced variables
print("\n" + "="*60)
print("TESTING OVERALL MODEL WITH REDUCED VARIABLES")
print("="*60)

# Prepare reduced dataset for modeling
X_reduced = df_reduced[all_keep_features].copy()
y_reduced = df_reduced['Class'].copy()

# Handle NAs
mask_reduced = ~X_reduced.isna().any(axis=1)
X_reduced = X_reduced[mask_reduced]
y_reduced = y_reduced[mask_reduced]

print(f"\nReduced dataset:")
print(f"  Samples: {len(X_reduced)}")
print(f"  Variables: {len(X_reduced.columns)}")

# Train Random Forest on full original variables
RF_full = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
RF_full.fit(X, y)
oa_full = RF_full.score(X, y)

# Train Random Forest on reduced variables
RF_reduced_final = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
RF_reduced_final.fit(X_reduced, y_reduced)
oa_reduced_final = RF_reduced_final.score(X_reduced, y_reduced)

print(f"\n=== Overall Model Comparison ===")
print(f"Full model:")
print(f"  Variables: {len(X.columns)}")
print(f"  Samples: {len(X)}")
print(f"  OA: {oa_full:.3f}")

print(f"\nReduced model:")
print(f"  Variables: {len(X_reduced.columns)}")
print(f"  Samples: {len(X_reduced)}")
print(f"  OA: {oa_reduced_final:.3f}")

print(f"\nChange in OA: {oa_reduced_final - oa_full:+.3f}")

if abs(oa_reduced_final - oa_full) < 0.05:
    print("✓ Reduction successful: OA maintained within 5%")
else:
    print("⚠ Warning: OA changed by more than 5%")

# Save reduced dataset to CSV
output_path = r"C:\Users\leila\Dropbox\MayoWetlands\wetland_alldata_2025_uscore_averaged_reduced.csv"
df_reduced.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"✓ Successfully saved reduced dataset to:")
print(f"  {output_path}")
print(f"{'='*60}")